In [ ]:
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/juhenes/ngiml"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/ngiml")

if REPO_DIR.exists():
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "origin", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "origin", REPO_BRANCH], check=True)
else:
    subprocess.run([
        "git",
        "clone",
        "--branch",
        REPO_BRANCH,
        "--single-branch",
        REPO_URL,
        str(REPO_DIR),
    ], check=True)

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
print(f"Repo ready at {REPO_DIR} on branch {REPO_BRANCH}")


In [ ]:
from pathlib import Path

from google.colab import drive

from tools.infer_helpers import get_model_complexity_stats, run_prepared_dataset_inference

drive.mount('/content/drive', force_remount=False)

CHECKPOINT_PATH = Path('/content/drive/MyDrive/thesis/ngiml-single/checkpoints/best_checkpoint.pt')
HF_DATASET_REPO_ID = 'juhenes/ngiml-test'
DRIVE_OUTPUT_ROOT = Path('/content/drive/MyDrive/thesis/ngiml-single-inference')
HF_SNAPSHOT_LOCAL_DIR = Path('/content/hf_datasets/ngiml_test')

INFERENCE_STRATEGY = 'direct'
THRESHOLD_FOR_METRICS = None
PLOT_BINARY_THRESHOLD = 0.5
DIRECT_BATCH_SIZE = 64

In [ ]:
run = run_prepared_dataset_inference(
    checkpoint_path=CHECKPOINT_PATH,
    hf_dataset_repo_id=HF_DATASET_REPO_ID,
    output_root=DRIVE_OUTPUT_ROOT,
    hf_snapshot_local_dir=HF_SNAPSHOT_LOCAL_DIR,
    inference_strategy=INFERENCE_STRATEGY,
    threshold_for_metrics=THRESHOLD_FOR_METRICS,
    plot_binary_threshold=PLOT_BINARY_THRESHOLD,
    direct_batch_size=DIRECT_BATCH_SIZE,
)

print('Snapshot:', run['snapshot_path'])
print('Device:', run['device'])
print('Normalization:', run['normalization_mode'])
print('Threshold for CSV metrics:', run['threshold_for_metrics'])
print('Plot threshold:', run['plot_binary_threshold'])
print('Direct batch size:', run['direct_batch_size'])
print('Saved full CSV:', run['results_csv'])
print('Saved summary CSV:', run['summary_csv'])
print('Saved plot root:', run['plot_output_dir'])
display(run['summary_df'])


In [ ]:
profile_input_size = int(run['checkpoint_info'].get('input_size', 448))
stats = get_model_complexity_stats(run['model'].cpu().eval(), input_size=(1, 3, profile_input_size, profile_input_size))

print('Checkpoint:', CHECKPOINT_PATH)
print('Input shape:', stats['input_size'])
print('Trainable params:', f"{stats['trainable_params']:,}")
print('Total params:', f"{stats['total_params']:,}")
print('MACs:', stats['macs'])
print('Approx FLOPs:', stats['flops'])
print('FLOPs source:', stats['flops_source'])
if stats.get('flops_error'):
    print('FLOPs note:', stats['flops_error'])
